1. KV Caching


In [1]:
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=False)
)

Now for the main event. We call model.generate().

Crucially, the use_cache=True argument is on by default for most autoregressive models. This is what enables KV caching. We are explicitly writing it here to make it clear, but you usually don't have to.

Stage 2: Autoregressive Generation (The Token-by-Token Loop)

Now, the model generates the rest of the text one token at a time. This is where the cache becomes critical.

- To generate the 1st new token:
    - The model only needs to process the last token of the prompt.
    - It calculates the Query (Q) for this token and uses it to attend to all the K and V vectors already in the cache.
    - After predicting the new token, it calculates the K and V for this new token only and appends them to the cache.

- To generate the 2nd new token:
    - The model only processes the 1st new token it just generated.
    - It calculates its Q vector and attends to the entire, updated cache (prompt tokens + 1st new token).
    - It then computes the K and V for the 2nd new token and appends them to the cache.

This loop continues, and at each step, the model avoids re-calculating K and V for all previous tokens. It's a massive computational saving!

The Magic of Library Abstraction

Libraries like Hugging Face Transformers abstract away this complex state management. By simply using model.generate(), we automatically get the benefits of KV Caching without needing to manually handle the past_key_values object at each step.

Final Takeaways

- What it is: A technique to store and reuse Key/Value vectors of past tokens during autoregressive generation.
- Why it's used: To dramatically reduce computation and lower latency (speed up generation time per token).
- The Trade-off: It consumes more memory (VRAM) because the cache grows linearly with every new token generated.


2: Observing Attention Configuration (MHA, MQA, GQA)

Goal: This demo shows how to inspect a model's configuration to determine its attention mechanism (Multi-Head, Multi-Query, or Grouped-Query Attention). Understanding this is key to predicting a model's memory usage for the KV Cache.


Step 3: Inspect Key Attention Attributes

The two most important attributes for determining the attention type are:
- num_attention_heads: The number of attention heads for the Query (Q) projections.
- num_key_value_heads: The number of attention heads for the Key (K) and Value (V) projections.

Extracted Attributes for 'meta-llama/Llama-3.2-1B':
  Number of Query Heads (N_q):         32
  Number of Key/Value Heads (N_kv):    8

Step 4: Determine the Attention Type

Now we can apply simple logic based on the two numbers we just extracted:
- If N_q == N_kv, it's Multi-Head Attention (MHA).
- If N_kv == 1, it's Multi-Query Attention (MQA).
- If 1 < N_kv < N_q, it's Grouped-Query Attention (GQA).

Final Interpretation

For the model meta-llama/Llama-3.2-1B, we have confirmed it uses Grouped-Query Attention (GQA).

Why this matters:
- Reduced Memory: A standard MHA model would have needed 32 sets of Key/Value heads in its KV Cache. By using only 8, GQA reduces the KV Cache size by a factor of 4 (32 / 8).
- Faster Inference: A smaller KV Cache means less data needs to be read from slow GPU memory (HBM) at each generation step, which reduces the memory bandwidth bottleneck and speeds up inference.
- Longer Context: The memory savings from GQA allow the model to handle longer sequences of text before running out of VRAM.


3: Measuring Memory Growth

This is the core of our experiment. We will loop through each of our specified generation_lengths. In each iteration, we will:
1. Reset Memory Stats: Use torch.cuda.reset_peak_memory_stats() to clear the memory counter. This is crucial for isolating the memory usage of each specific model.generate() call.
2. Generate Text: Run model.generate() with use_cache=True.
3. Record Peak Memory: Immediately after generation, record the peak memory usage for that run.
4. Clean Up: Clear the generated tensors to ensure each run is as independent as possible.

Generated Tokens | Total Peak Memory (MB)
-----------------|------------------------
              50 |                9446.75
             100 |                9449.88
             200 |                9456.13
             300 |                9462.38
             400 |                9470.79

Step 5: Interpretation and Conclusion

The results clearly show a linear relationship between the number of generated tokens and the peak GPU memory usage.

- Baseline Memory: The initial memory consumed after loading the model is the cost of storing the model's weights themselves.
- Memory Growth: The subsequent increase in memory with each generation run is almost entirely due to the KV Cache. For every new token generated, its corresponding Key and Value vectors are computed and stored in the cache for all attention heads across all layers.

Calculating Memory per Token

We can estimate the memory cost per token by looking at the slope of our graph. Let's calculate it between the first and last data points:

Plot saved to kv_cache_memory_growth.png

Final Takeaway

This exercise empirically proves the memory cost of KV Caching. While it's essential for fast inference, it's also a primary constraint on the maximum sequence length a model can handle on a given GPU. This is why techniques like Grouped-Query Attention (GQA), which we'll cover next, were developed — to reduce the size of the KV Cache and mitigate this exact problem.


GQA Attention


--- Analyzing Llama-3.2-1B (GQA): meta-llama/Llama-3.2-1B ---
  Number of Layers (L):        16
  Query Heads (N_q):             32
  Key/Value Heads (N_kv):        8
  Head Dimension (D_head):       64
  Data Type Size (bytes):      4 (torch.float32)
  Identified Attention Type:     Grouped-Query Attention (GQA)

--- Analyzing GPT-2 XL (MHA): openai-community/gpt2-xl ---
    Number of Layers (L):        48
    Query Heads (N_q):             25
    Key/Value Heads (N_kv):        25
    Head Dimension (D_head):       128
    Data Type Size (bytes):      4 (torch.float32)
    Identified Attention Type:     Multi-Head Attention (MHA)

Step 4: Compare Results and Quantify Savings

Logic: Now we will process the dictionaries returned by our analysis function to present a clear, side-by-side comparison. We will focus on two key metrics:
1. Per-Layer Cache Size: This normalizes the comparison by ignoring the total number of layers, allowing us to see the direct architectural advantage of GQA vs. MHA.
2. Internal Saving Factor: For the Llama model, we calculate how much memory it saves with GQA compared to a hypothetical version of itself with MHA. This isolates the benefit of GQA within a single architecture.


--- DETAILED ANALYSIS & COMPARISON ---

--- Direct Comparison (Per Layer, Per Token KV Cache) ---
  Llama-3.2-1B (GQA): 4.00 KB/layer/token
  GPT-2 XL (MHA):     12.50 KB/layer/token

--- GQA Internal Saving Factor (for Llama-3.2-1B) ---
  Llama-3.2-1B uses 8 KV heads for 32 Query heads.
  This provides an internal KV cache saving factor of 4.00x compared to if it used MHA.

Final Summary and Conclusion

This exercise quantified and compared the theoretical KV cache memory usage for meta-llama/Llama-3.2-1B (GQA) and openai-community/gpt2-xl (MHA).

Model 1: meta-llama/Llama-3.2-1B (GQA)
- Configuration: L=16, N_q=32, N_kv=8, D_head=64, dtype=float16 (2 bytes).
- Attention Mechanism: Identified as Grouped Query Attention (GQA) with a Grouping Factor of 4 (32 Query heads / 8 KV heads).
- KV Cache Memory:
    - Per Layer, Per Token: 2.00 KB (calculated as 2 * 8 * 64 * 2 bytes).
    - Total Per Token (all 16 layers): 0.0313 MB.

Model 2: openai-community/gpt2-xl (MHA)
- Configuration: L=48, N_q=25, N_kv=25, D_head=64, dtype=float16 (2 bytes).
- Attention Mechanism: Identified as Multi-Head Attention (MHA).
- KV Cache Memory:
    - Per Layer, Per Token: 6.25 KB (calculated as 2 * 25 * 64 * 2 bytes).
    - Total Per Token (all 48 layers): 0.2930 MB.

Direct Comparison and Savings
- Per-Layer Efficiency: The per-layer KV cache for Llama-3.2-1B (GQA) at 2.00 KB is significantly smaller than GPT-2 XL's MHA cache at 6.25 KB. This demonstrates that GQA's architecture is inherently more memory-efficient at the layer level, primarily due to using far fewer K/V heads (8 vs. 25).
- Internal Saving Factor: Llama-3.2-1B's GQA provides a 4x memory saving for its KV cache compared to if it had used a traditional MHA design (32 / 8 = 4).

Practical Implication (Max Sequence Length with 6GB VRAM for Cache)
- Llama-3.2-1B (GQA): Could theoretically support a sequence of ~196,608 tokens.
- GPT-2 XL (MHA): Could theoretically support a sequence of ~20,971 tokens.

Conclusion

The exercise successfully demonstrates that Grouped-Query Attention is a highly effective technique for reducing the memory footprint of the KV Cache. By using fewer Key/Value heads than Query heads, GQA significantly lowers the amount of data that must be stored for each generated token. This architectural improvement, as shown by the comparison, directly translates into a much larger maximum context length, enabling modern models like Llama-3.2-1B to handle longer sequences far more efficiently than older MHA-based models like GPT-2 XL.
